# 一、前言

这一节 dropout 是从"网络结构"角度正则化——训练时随机丢神经元，逼网络不依赖任何单点。

# 二、代码

## 1.从零实现 dropout_layer

In [14]:
import torch
from torch import nn
from d2l import torch as d2l

def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1   # 断言检查。如果传入的丢弃概率不在 0‑1 区间，程序直接报错，防止非法参数。
    if dropout == 1:           # 边界处理 dropout==1 返回全零（除零保护：1-1=0 会除零）丢弃概率等于 1 等价于所有神经元全部丢掉
        return torch.zeros_like(X)
    if dropout == 0:           # dropout==0 直接返回（省一次随机采样）; 一个神经元都不丢弃，Dropout 不起任何作用。
        return X 
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

dropout_layer(X,dropout)：手动实现丢弃法（Dropout）正则化，在训练时随机把一部分神经元置 0，用来缓解神经网络过拟合。
X：输入张量（神经网络某一层的激活输出）
dropout：丢弃概率，取值 [0,1]
若 dropout = 0.3：30% 的神经元被丢弃（置零）；剩下 70% 保留

torch.rand(X.shape) 生成一个和 X 相同形状的张量，每个元素是0~1 之间均匀分布的随机数。
随机数 > dropout 后为 True（保留这个神经元） 的概率是 1-dropout，
随机数 ≤ dropout → False（丢弃，置 0）0 代表丢弃神经元。
.float() 转成 0/1；得到布尔 mask，mask 称为掩码矩阵：由 0 和 1 组成。1 代表保留神经元；这就是"以概率 p 置零"的随机实现


关键一行：mask * X / (1.0 - $dropout$)。 除以 (1-p) 叫"无偏缩放"——期望 E[h'] = (1-p)* h /(1-p) + p*0 = h，激活值期望在 dropout 前后不变。如果不除，训练时每层激活期望只剩 (1-p) 倍，网络要额外学一个缩放，训练测试行为不一致，性能掉。

为什么必须除以 1-dropout（缩放补偿，Inverted Dropout）
如果不做除法：假如训练时 70% 神经元激活；测试阶段 Dropout 关闭，全部神经元激活，输出期望就变大了，训练、测试输出分布不一致！

## 2.验证函数行为

In [15]:
X = torch.arange(16, dtype = torch.float).reshape((2, 8))
print(X)
print(dropout_layer(X, 0.))
print(dropout_layer(X, 0.5))
print(dropout_layer(X, 1.))

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  0.,  0.,  6.,  8.,  0.,  0., 14.],
        [16., 18., 20.,  0.,  0.,  0.,  0.,  0.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


用 16 个连续数验证三个边界行为——p=0 原样返回、p=1 全零、p=0.5 大约一半元素被置零且保留元素翻倍（除以 0.5）。
观察： p=0.5 时保留元素数值 ×2，这就是缩放的效果。多跑几次会发现置零的位置每次不同——随机性来自 torch.rand。

## 3.从零实现：带 dropout 的 MLP

In [16]:
dropout1 = 0.2  # 第一层隐藏层丢弃概率 20%，保留概率 80%
dropout2 = 0.5  # 第二层隐藏层丢弃概率 50%，保留概率 50%
# 丢弃概率越大，随机关掉的神经元越多，正则强度越强。

class Net(nn.Module):
    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2, is_training = True):
        super(Net, self).__init__()  # 调用父类nn.Module的构造函数，固定写法，不能省略
        self.num_inputs = num_inputs  # 保存输入特征维度
        self.training = is_training
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)   #输入层→隐藏层1
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2) #隐藏层1→隐藏层2
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)  #隐藏层2→输出层
        self.relu = nn.ReLU()

    def forward(self, X):     # 前向传播函数
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))
        if self.training == True:   # 只有训练的时候，才对 H1 执行 Dropout
            H1 = dropout_layer(H1, dropout1)
        H2 = self.relu(self.lin2(H1))   # H1 送入第二层全连接层，经过 ReLU 激活，得到第二层隐藏输出 H2。
        if self.training == True:
            H2 = dropout_layer(H2, dropout2)
        out = self.lin3(H2)   # 最后一层输出层 不加 Dropout！，直接输出，返回预测结果。
        return out
net = Net(784, 10, 256, 256)

dropout 放在激活函数之后（ReLU → dropout）：d2l 的标准顺序。ReLU 已经把负数压成 0，dropout 再随机丢一部分正值。

self.training = is_training 这一行是个暗坑：它覆盖了 nn.Module 自带的 self.training 属性。好处是 net.train() / net.eval() 仍然生效（nn.Module.train(mode) 内部就是写 self.training = mode），所以 forward 里 if self.training == True 能被训练/评估模式正确驱动。如果不写这行、用 nn.Module 默认的 self.training，也可以 ，d2l 这么写只是为了教学显式。<br>
self.training = True → 开启 Dropout（训练阶段）<br>
self.training = False → 关闭 Dropout（测试 / 推理阶段）

nn.Module 默认就是训练模式——Module.$__init__$ 里本身就设了 self.training = True。所以 d2l 从零版默认 is_training=True 其实是冗余的（跟默认值一致），写出来纯粹为了教学显式。
d2l 原版训练循环里甚至不显式调 net.train()，靠的就是默认训练模式；只在 evaluate_accuracy 里调 net.eval()。我给你的教案里两个都显式写，是更稳的习惯——你以后写自定义模型，不显式 net.train() 可能踩默认值陷阱。

两个隐藏层 dropout 概率不同：靠近输入层 0.2，靠近输出层 0.5。为什么？输入侧特征还比较"原始"，丢多了信息损失大；深层特征冗余度高，可以多丢。这个经验值后面练习会让你验证交换概率会发生什么。
网络结构：<br>
输入 → 全连接 1 → ReLU → Dropout1 → 全连接 2 → ReLU → Dropout2 → 全连接 3 →输出<br>
两层隐藏层的多层感知机（MLP）

H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))<br>
X.reshape((-1, self.num_inputs))：把图片展平。以 Fashion‑MNIST 为例：原图 28×28，展平成 784 维向量；‑1代表自动推算批次大小。<br>
self.lin1()：线性变换 XW+b<br>
self.relu()：激活函数，得到第一层隐藏输出H1

## 4.训练循环

In [17]:
num_epochs, lr, batch_size = 10, 0.5, 256
loss = nn.CrossEntropyLoss(reduction = 'none')
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
trainer = torch.optim.SGD(net.parameters(), lr = lr)
def train_epoch(net, train_iter, loss, trainer):
    net.train()                      # 打开 dropout
    metric = d2l.Accumulator(3)
    for X, y in train_iter:
        trainer.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.mean().backward()
        trainer.step()
        metric.add(float(l.sum()), d2l.accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2], metric[1] / metric[2]

def evaluate_accuracy(net, data_iter):
    net.eval()                      # 关掉 dropout
    metric = d2l.Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(d2l.accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(net, train_iter, loss, trainer)
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}, loss {train_loss:.3f}, '
          f'train acc {train_acc:.3f}, test acc {test_acc:.3f}')

epoch 1, loss 0.943, train acc 0.652, test acc 0.797
epoch 2, loss 0.541, train acc 0.799, test acc 0.794
epoch 3, loss 0.475, train acc 0.827, test acc 0.824
epoch 4, loss 0.433, train acc 0.841, test acc 0.821
epoch 5, loss 0.406, train acc 0.852, test acc 0.856
epoch 6, loss 0.387, train acc 0.860, test acc 0.844
epoch 7, loss 0.378, train acc 0.861, test acc 0.862
epoch 8, loss 0.362, train acc 0.867, test acc 0.850
epoch 9, loss 0.349, train acc 0.871, test acc 0.865
epoch 10, loss 0.340, train acc 0.875, test acc 0.835


net.train() / net.eval() 是这节最容易踩的坑。 训练函数开头必须 net.train()（打开 dropout），评估函数开头必须 net.eval()（关掉 dropout）。不写 net.eval()，你的"测试准确率"其实是在 dropout 随机扰动下测的，数字每次跑都不一样，且偏低。 这是 4.6 最经典的学生 bug。

l.mean().backward()：loss 用 reduction='none' 保留逐样本损失，自己控制 mean——这正是你 ch3 修过的 mean/sum 梯度 bug 的延续，这里保持一致。

metric.add(float(l.sum()), ...)：Accumulator 累加 loss 总和、正确数、样本数，最后相除得均值。和你 ch3 的 train_epoch 同构，只是多了 net.train()。

## 5.简洁实现

In [20]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, 256),
                    nn.ReLU(),
                    nn.Dropout(dropout1),
                    nn.Linear(256, 256),
                    nn.ReLU(),
                    nn.Dropout(dropout2),
                    nn.Linear(256, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std = 0.01)
net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): Dropout(p=0.5, inplace=False)
  (7): Linear(in_features=256, out_features=10, bias=True)
)

nn.Dropout(p) 内部自动处理了两件事：随机置零（默认用 inverted dropout，训练时除以 1-p）+ 训练/测试模式自动切换（它读 self.training 标志，net.eval() 时直接透传，什么都不做）。所以简洁版 forward 里不需要写任何 if。

nn.Flatten() 显式展平：因为 nn.Sequential 里没有自定义 forward，必须把 28×28 展平成 784。

nn.Dropout 的位置同样是 ReLU 之后、下一层 Linear 之前。

net.apply(init_weights) 递归地对所有子模块应用初始化

## 6.简洁版训练

In [21]:
trainer = torch.optim.SGD(net.parameters(), lr = lr)
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(net, train_iter, loss, trainer)
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}, loss {train_loss:.3f}, '
          f'train acc {train_acc:.3f}, test acc {test_acc:.3f}')

epoch 1, loss 1.111, train acc 0.571, test acc 0.755
epoch 2, loss 0.583, train acc 0.784, test acc 0.710
epoch 3, loss 0.491, train acc 0.821, test acc 0.801
epoch 4, loss 0.439, train acc 0.839, test acc 0.837
epoch 5, loss 0.413, train acc 0.850, test acc 0.838
epoch 6, loss 0.395, train acc 0.855, test acc 0.854
epoch 7, loss 0.375, train acc 0.863, test acc 0.836
epoch 8, loss 0.364, train acc 0.867, test acc 0.849
epoch 9, loss 0.354, train acc 0.870, test acc 0.858
epoch 10, loss 0.339, train acc 0.874, test acc 0.843


# 三、小结

1.为什么 dropout_layer 里要除以 (1.0 - dropout)？如果去掉这行，训练时第 k 个隐藏层的激活值期望会被缩小多少倍？测试时会发生什么？<br>
如果不除以他的话，那么当在测试时，X 是没有被置零的，而训练的时候，是有为 0 被抛弃的输入的，那么则会导致当前层输出的平均值(即所谓的期望)小于测试时的，那么得出的 y_hat 值的差距就会变大。每一层被缩小 (1 - dropout) 倍。导致测试时的 loss 与训练时的差距比较大<br>
但期望相等是大量采样后的平均效果；测试时单张样本只执行一次前向传播<br>
第 k 层的累积效应。如果每层都 dropout，第 k 个隐藏层的激活期望沿线性路径近似乘了 (1-p)ᵏ（每过一层 dropout 就乘一次 1-p；ReLU 是非线性的，所以是"近似"）。层数越深，偏差越是指数放大——这就是为什么说"层数越深误差越放大"。<br>
不除以 (1-p) 的对策不是只能崩——你可以在测试时把激活乘 (1-p) 来补偿。这就是经典 dropout（Srivastava 2014）的做法：训练不缩放、测试缩放。d2l 用的是 inverted dropout（训练时缩放），好处是测试前向干净、不用额外操作。两种在期望上等价

2.简洁实现里，如果测试时忘了调 net.eval()，你预计测试准确率会有什么表现？为什么？<br>
如果不调的话，就是没关 dropout ，那么仍然还是相当于训练模式，每次前向都是随机子网络 → 预测不稳定、不准,有些神经元就是关闭的，虽然还是除以了(1.0 - dropout)，但却只迭代一次，训练时每步一个随机 mask，测试时用全网络 ≈ 对指数多个子网络做平均（集成）；测试时开 dropout = 只抽了一个子网络，既没平均、又没完整容量。，那么这个预测就是不准的，整体准确率会偏低，没有起到预测的作用

3.把 dropout1 和 dropout2 交换（浅层 0.5、深层 0.2），预计训练曲线和最终准确率会怎么变？给出你的理由（先猜想，再跑完验证）。<br>
浅层 0.5 意味着输入侧原始特征直接丢一半，信息损失发生在最关键的位置 → 训练更难、准确率更低；<br>
训练曲线会更抖、收敛更慢（不只是"损失更大"）；最终 test acc 通常也比默认配置低（不只是训练 acc）

4.从零版 Net 里 self.training = is_training 覆盖了 nn.Module 自带的 training 属性——net.train() 和 net.eval() 为什么仍然能正确开关 dropout？<br>
net.train() / net.eval() 的本质就是给 self.training 这个属性赋值。所以在 $__init__$ 里写 self.training = is_training，只是改了这个属性的初始值；而 forward 里的 if self.training == True 读的是同一个属性。调 net.train() 时，它把这个属性重新赋成 True；调 net.eval() 时重新赋成 False——开关就是这么生效的，跟初始值写什么都没关系。

5.(概念)用一句话解释：为什么 dropout 能缓解过拟合，但 dropout 概率设太大反而会欠拟合？<br>
dropout 是正则化器——它抬高训练误差、压小泛化差距（train-test 差距），从而缓解过拟合，丢弃概率越大，训练时被关掉的神经元越多，模型能学到东西的能力被人为削弱；约束太强，模型连训练集本身的规律都拟合不上 → 产生欠拟合。此时是欠拟合，不是过拟合。

# 附加

dropout 的动机全部来自 bias-variance tradeoff 和过拟合。核心对应关系：<br>
偏差-方差权衡 → d2l 4.6 开头"线性模型高偏差低方差 vs 深度网络低偏差高方差"<br>
过拟合（模型记住训练集噪声）→ d2l 提到的 2017 年"随机标签过拟合"实验：深度网络能把随机标签训练集记到 100%，泛化差距 90%

Dropout 为什么有效，三个视角（都要能说清）：<br>
1. 破坏共适应（原论文视角）：相邻层神经元协同"勾结"记住训练集，dropout 随机拆散它们，逼每个神经元独立有用。<br>
2. 隐式集成（直觉视角）：每次迭代 dropout 的 mask 不同 ≈ 训练了指数多个不同的子网络，测试时用全网络 ≈ 对这些子网络做平均（bagging 思想）。<br>
3. 平滑性（数学视角）：Bishop 1995 证明"输入加噪声训练 ≈ Tikhonov 正则化"，dropout 把噪声注入内部层，等价于要求函数对扰动不敏感。<br>
一个必须记住的边界：dropout 只在训练时用。测试时用全网络给出确定性预测。例外：有人测试时故意开 dropout 来估计预测不确定性（test-time dropout）——d2l 原文提了一句，知道即可。

# 测试

In [22]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, 256),
                    nn.ReLU(),
                    nn.Dropout(dropout2),
                    nn.Linear(256, 256),
                    nn.ReLU(),
                    nn.Dropout(dropout1),
                    nn.Linear(256, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std = 0.01)
net.apply(init_weights)

trainer = torch.optim.SGD(net.parameters(), lr = lr)
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(net, train_iter, loss, trainer)
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}, loss {train_loss:.3f}, '
          f'train acc {train_acc:.3f}, test acc {test_acc:.3f}')

epoch 1, loss 1.268, train acc 0.512, test acc 0.692
epoch 2, loss 0.641, train acc 0.757, test acc 0.721
epoch 3, loss 0.545, train acc 0.798, test acc 0.783
epoch 4, loss 0.491, train acc 0.820, test acc 0.823
epoch 5, loss 0.470, train acc 0.827, test acc 0.832
epoch 6, loss 0.449, train acc 0.836, test acc 0.793
epoch 7, loss 0.435, train acc 0.840, test acc 0.850
epoch 8, loss 0.424, train acc 0.844, test acc 0.825
epoch 9, loss 0.414, train acc 0.848, test acc 0.835
epoch 10, loss 0.403, train acc 0.853, test acc 0.853
